In [ ]:
run_date = None
config_path = 'config/settings.yaml'


In [ ]:
%matplotlib inline
from pathlib import Path
import warnings
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=0.9)

from qis_risk_report.data.loaders import load_returns, load_portfolio, load_weights, load_factors
from qis_risk_report.risk import metrics as m
from qis_risk_report.risk.attribution import rolling_ols_attribution, contribution_decomposition
from qis_risk_report.risk.scenarios import default_scenarios, replay_scenario, run_scenario, ShockParams, HistoricalScenario
from qis_risk_report.risk.portfolio import marginal_var, component_var, portfolio_var, qis_portfolio_correlation, diversification_benefit
from qis_risk_report.reports import charts

cfg = yaml.safe_load(Path(config_path).read_text(encoding='utf-8-sig'))
returns_df = load_returns(cfg['data']['returns_path'])
portfolio_returns_df = load_portfolio(cfg['data']['portfolio_returns_path'])
weights_df = load_weights(cfg['data']['weights_path'])
factors_df = load_factors(cfg['data']['factors_path'])

weights = weights_df.iloc[-1]
sub_cols = [c for c in returns_df.columns if c != 'total']
port_cols = list(portfolio_returns_df.columns)
port_weights = weights.reindex(port_cols).fillna(0)

_rf = float(cfg.get('risk_free_rate', 0.0))
_run_date = run_date or returns_df.index[-1].strftime('%Y-%m-%d')
_run_dt = pd.Timestamp(_run_date)
_ytd_start = _run_dt.replace(month=1, day=1)
_mtd_start = _run_dt.replace(day=1)

print(f'Report date: {_run_date}  |  QIS: {returns_df.index[0].date()} to {returns_df.index[-1].date()}  |  {len(returns_df)} trading days')


# QIS Risk Report

---


## KPI Summary


In [ ]:
_s = returns_df['total']
_s252 = _s.iloc[-252:]

_ytd_s = _s.loc[_ytd_start:_run_date]
_ytd_ret = float((1 + _ytd_s).prod() - 1) if len(_ytd_s) > 0 else float('nan')
_1y_ann = m.annualised_return(_s252) if len(_s252) >= 2 else float('nan')
_ann_vol_1y = m.annualised_volatility(_s252)
_sharpe_1y = (_1y_ann - _rf) / _ann_vol_1y if _ann_vol_1y > 0 else float('nan')
_max_dd = m.max_drawdown(_s)
_curr_dd = float(m.drawdown_series(_s).iloc[-1])
_var99_1d = m.historical_var(_s252, confidence=0.99)

kpi = pd.DataFrame({
    'KPI': [
        'YTD Return',
        '1Y Annualised Return',
        'Annualised Volatility (1Y)',
        'Sharpe Ratio (1Y)',
        'Maximum Drawdown',
        'Current Drawdown',
        '1-Day VaR 99%',
    ],
    'Value': [
        f'{_ytd_ret:.2%}',
        f'{_1y_ann:.2%}',
        f'{_ann_vol_1y:.2%}',
        f'{_sharpe_1y:.2f}',
        f'{_max_dd:.2%}',
        f'{_curr_dd:.2%}',
        f'{_var99_1d:.3%}',
    ],
}).set_index('KPI')
display(kpi)


---

## 1. Performance


In [ ]:
def _period_return(s, start, end):
    w = s.loc[start:end]
    return float((1 + w).prod() - 1) if len(w) > 0 else float('nan')

rows = []
for col in returns_df.columns:
    s = returns_df[col]
    _1y_data = s.iloc[-252:]
    rows.append({
        'Component': col,
        '1D': f"{float(s.iloc[-1]):.2%}",
        'MTD': f"{_period_return(s, _mtd_start, _run_date):.2%}",
        'YTD': f"{_period_return(s, _ytd_start, _run_date):.2%}",
        '1Y': f"{float((1 + _1y_data).prod() - 1):.2%}",
        '1Y Ann.': f"{m.annualised_return(_1y_data):.2%}" if len(_1y_data) >= 2 else 'N/A',
        'Inception': f"{float((1 + s).prod() - 1):.2%}",
    })
display(pd.DataFrame(rows).set_index('Component'))


In [ ]:
fig = charts.plot_cumulative_return(returns_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_monthly_heatmap(returns_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_daily_return_bars(returns_df, window=63)
plt.tight_layout()
plt.show()


---

## 2. Risk Metrics


In [ ]:
rows = []
for col in returns_df.columns:
    s = returns_df[col]
    s252 = s.iloc[-252:]
    rows.append({
        'Component': col,
        'Ann Vol 252d': f'{m.annualised_volatility(s252):.2%}',
        'Ann Vol All': f'{m.annualised_volatility(s):.2%}',
        'Sharpe 252d': f'{m.sharpe_ratio(s252, _rf):.2f}',
        'Sharpe All': f'{m.sharpe_ratio(s, _rf):.2f}',
        'Sortino 252d': f'{m.sortino_ratio(s252, _rf):.2f}',
        'Sortino All': f'{m.sortino_ratio(s, _rf):.2f}',
        'Max DD': f'{m.max_drawdown(s):.2%}',
        'Curr DD': f'{float(m.drawdown_series(s).iloc[-1]):.2%}',
        'Max DD Dur (d)': m.drawdown_duration(s),
        'VaR 95% (252d)': f'{m.historical_var(s252, 0.95):.3%}',
        'VaR 99% (252d)': f'{m.historical_var(s252, 0.99):.3%}',
        '10d VaR 99%': f'{m.historical_var(s252, 0.99, 10):.3%}',
        'CVaR 95% (252d)': f'{m.expected_shortfall(s252, 0.95):.3%}',
    })
display(pd.DataFrame(rows).set_index('Component'))


In [ ]:
fig = charts.plot_rolling_volatility(returns_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_underwater(returns_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_return_distribution(returns_df, confidence=0.95)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_correlation_heatmap(returns_df)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_rolling_pairwise_correlation(returns_df, window=252)
plt.tight_layout()
plt.show()


---

## 3. Factor Attribution


In [ ]:
_attr_252 = {col: rolling_ols_attribution(returns_df[col], factors_df, window=252) for col in sub_cols}
ytd_rets = returns_df.loc[_ytd_start:_run_date]
ytd_factors = factors_df.loc[_ytd_start:_run_date]

rows = []
for col in sub_cols:
    attr = _attr_252[col]
    latest = attr.iloc[-1]
    betas = latest.drop('r_squared')
    fc_contribs = {
        fc: float(betas.get(fc, 0.0) * ytd_factors[fc].sum())
        for fc in factors_df.columns
    }
    total_ytd = float(ytd_rets[col].sum()) if col in ytd_rets else float('nan')
    alpha = total_ytd - sum(fc_contribs.values())
    row = {'Component': col, 'R²': f"{float(latest['r_squared']):.3f}"}
    for fc in factors_df.columns:
        row[f'β_{fc}'] = f"{float(betas.get(fc, 0.0)):.3f}"
    for fc in factors_df.columns:
        row[f'YTD {fc}'] = f"{fc_contribs[fc]:.2%}"
    row['Alpha (YTD)'] = f"{alpha:.2%}"
    rows.append(row)
display(pd.DataFrame(rows).set_index('Component'))


In [ ]:
for col in sub_cols:
    fig = charts.plot_rolling_factor_betas(returns_df[col], factors_df, window=252)
    plt.tight_layout()
    plt.show()


In [ ]:
fig = charts.plot_ytd_attribution_bar(returns_df, factors_df, _run_date, window=252)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_contribution_area(returns_df, window=252)
plt.tight_layout()
plt.show()


---

## 4. Scenario Analysis


In [ ]:
_all_scenarios = default_scenarios()

rows = []
for scen in _all_scenarios:
    try:
        result = replay_scenario(scen, returns_df)
        window = returns_df.loc[scen.start:scen.end]
        vol = m.annualised_volatility(window['total']) if ('total' in window.columns and len(window) >= 2) else float('nan')
        row = {
            'Event': scen.name,
            'Start': scen.start,
            'End': scen.end,
            'Days': result.metadata.get('n_days', 0),
            'Strategy P&L': f'{result.pnl_total:.2%}',
        }
        for k, v in result.pnl_by_component.items():
            row[k] = f'{v:.2%}'
        row['Ann. Vol'] = f'{vol:.2%}'
        rows.append(row)
    except Exception as exc:
        rows.append({'Event': scen.name, 'Start': scen.start, 'End': scen.end, 'Days': 0, 'Strategy P&L': f'N/A ({exc})'})

_scenario_results = []
for scen in _all_scenarios:
    try:
        _scenario_results.append(replay_scenario(scen, returns_df))
    except Exception:
        pass

display(pd.DataFrame(rows).set_index('Event'))


In [ ]:
fig = charts.plot_scenario_impact(_scenario_results)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_stress_path('GFC', returns_df, _all_scenarios)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_stress_path('COVID Crash', returns_df, _all_scenarios)
plt.tight_layout()
plt.show()


In [ ]:
_spot_shocks = cfg.get('synthetic_shocks', {}).get('spot_shocks', [-0.05, -0.025, 0.0, 0.025, 0.05])
_vol_shocks = cfg.get('synthetic_shocks', {}).get('vol_shocks', [-0.20, 0.0, 0.20])

shock_grid = {}
for spot in _spot_shocks:
    row_data = {}
    for vol in _vol_shocks:
        result = run_scenario(returns_df[sub_cols], ShockParams(spot_shock=spot, vol_shock=vol))
        row_data[f'vol {vol:+.0%}'] = f'{result.pnl_total:.2%}'
    shock_grid[f'spot {spot:+.1%}'] = row_data

display(pd.DataFrame(shock_grid).T)


---

## 5. Portfolio Risk Contribution


In [ ]:
_qis_weight = float(port_weights.get('qis_total', 0.0))
_qis_ret = returns_df['total'].reindex(portfolio_returns_df.index).dropna()
_qis_var95 = m.historical_var(_qis_ret.iloc[-252:], confidence=0.95) if len(_qis_ret) >= 252 else m.historical_var(_qis_ret, confidence=0.95)

_mv = marginal_var(portfolio_returns_df, port_weights, confidence=0.95)
_cv = component_var(portfolio_returns_df, port_weights, confidence=0.95)
_cv_total = float(_cv.sum())
_qis_mv = float(_mv.get('qis_total', float('nan')))
_qis_cv = float(_cv.get('qis_total', float('nan')))
_qis_cv_share = _qis_cv / _cv_total if _cv_total != 0 else float('nan')

_qis_corr_all = qis_portfolio_correlation(returns_df[['total']], portfolio_returns_df, port_weights)
_qis_port_corr = float(_qis_corr_all.get('total', float('nan')))

try:
    _div_b = diversification_benefit(portfolio_returns_df, port_weights, confidence=0.95)
    _div_b_pct = _div_b / _qis_var95 if _qis_var95 > 0 else float('nan')
except ValueError:
    _div_b = float('nan')
    _div_b_pct = float('nan')

port_summary = pd.DataFrame({
    'Metric': [
        'QIS Weight in Portfolio',
        'Standalone 1-Day VaR 95% (QIS)',
        'Marginal VaR Contribution of QIS',
        'Component VaR Share of QIS (% of total portfolio VaR)',
        'Trailing 252-Day Correlation (QIS vs. Portfolio)',
        'Diversification Benefit (standalone − component, % of standalone)',
    ],
    'Value': [
        f'{_qis_weight:.2%}',
        f'{_qis_var95:.3%}',
        f'{_qis_mv:.4%}',
        f'{_qis_cv_share:.1%}',
        f'{_qis_port_corr:.3f}',
        f'{_div_b_pct:.1%}',
    ],
}).set_index('Metric')
display(port_summary)


In [ ]:
_cv_full = component_var(portfolio_returns_df, port_weights, confidence=0.95)
_qis_sub_eq_w = pd.Series(1.0 / len(sub_cols), index=sub_cols)
_cv_qis_subs = component_var(returns_df[sub_cols], _qis_sub_eq_w, confidence=0.95)
_cv_qis_subs_scaled = _cv_qis_subs * _qis_weight

fig = charts.plot_component_var_decomposition(_cv_full, _cv_qis_subs_scaled)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_rolling_qis_portfolio_correlation(returns_df, portfolio_returns_df, port_weights, window=252)
plt.tight_layout()
plt.show()


In [ ]:
fig = charts.plot_diversification_benefit_over_time(portfolio_returns_df, weights_df, window=252, confidence=0.95)
plt.tight_layout()
plt.show()
